# Project: E-Commerce Analytics Pipeline
A complete data pipeline that processes ecommerce data, creates, clean data models, and generates business reports.

## Step 1: Setup
- Import libraries
- Create catalog: ecommerce, schemas: bronze, silver, gold

### Dataset
This is the dataset that is being used now:
https://www.kaggle.com/datasets/carrie1/ecommerce-data

In [0]:
# Importing and downloading the dataset
!pip install kagglehub
import kagglehub

# Download latest version to a specified path
path = kagglehub.dataset_download("carrie1/ecommerce-data", output_dir="/Workspace/Users/diego.freiregit@outlook.com/Learning/Sandhya - Zero to Hero Concepts/tmp/ecommerce-data")

print("Path to dataset files:", path)

df = spark.read.format("csv").option("header", "true").load(path)
display(df)

In [0]:
display(df.describe())

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce;

CREATE SCHEMA IF NOT EXISTS ecommerce.bronze;
CREATE SCHEMA IF NOT EXISTS ecommerce.silver;
CREATE SCHEMA IF NOT EXISTS ecommerce.gold;

# Step 2: Bronze Layer
Insert the raw data in the bronze layer (data as-is)

In [0]:
from pyspark.sql.functions import current_timestamp, col, current_date

df_bronze = (
    df
    .withColumn('_load_date', current_date())
    .withColumn('_load_time', current_timestamp())
    .withColumn('_source_file', col('_metadata.file_path'))
)

(
    df_bronze
    .write
    .format('delta')
    .mode('append')
    .saveAsTable('ecommerce.bronze.raw_invoices')
)

# Step 3: Silver Layer
Clean and validate the data and insert into the Silver table.

In [0]:
# Python/Spark implementation of the Silver Layer

from pyspark.sql.functions import col, expr, try_to_timestamp, lit
from delta.tables import DeltaTable

bronze_df = spark.table('ecommerce.bronze.raw_invoices')
max_load_time = bronze_df.agg({"_load_time": "max"}).collect()[0][0] or "1900-01-01 00:00:00"
print(f"Max load time from the bronze table: {max_load_time}")

# Creating the silver table if it doesn't exists
if not spark.catalog.tableExists("ecommerce.silver.invoicesSpark"):
    silver_df = (
        bronze_df
        .filter(
            (col("_load_time") == max_load_time) &
            (col("Quantity").cast("int") > 0) &
            (col("UnitPrice").cast("double") > 0) &
            col("CustomerID").isNotNull()
        )
        .select(
            col("InvoiceNo").cast("int").alias("invoice_no"),
            col("StockCode").cast("string").alias("stock_code"),
            col("Description").cast("string").alias("description"),
            col("Quantity").cast("int").alias("quantity"),
            try_to_timestamp(col("InvoiceDate"), lit("M/d/yyyy H:mm")).alias("invoice_date"),
            col("UnitPrice").cast("double").alias("unit_price"),
            col("CustomerID").cast("string").alias("customer_id"),
            col("Country").cast("string").alias("country"),
            col("_load_time")
        )
    )

    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("ecommerce.silver.invoicesSpark")
    )
    print("Table successfully created.")
else: # If the table does exists, merging the rows instead of creating a new one
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number

    # Getting the latest row for each invoice, in case there are multiple rows for the same invoice
    print("Table already exists. Merging the data instead")
    source_df = (
      bronze_df
      .filter(
        (col("_load_time") == max_load_time) &
        (col("Quantity").cast("int") > 0) &
        (col("UnitPrice").cast("double") > 0) &
        col("CustomerID").isNotNull()
      )
      .withColumn(
        "rn", row_number().over(
          Window.partitionBy("InvoiceNo","StockCode","CustomerID").orderBy(col("_load_time").desc())
          )
        )
      .filter(col("rn") == 1)
      .select(
        col("InvoiceNo").cast("int").alias("invoice_no"),
        col("StockCode").cast("string").alias("stock_code"),
        col("Description").cast("string").alias("description"),
        col("Quantity").cast("int").alias("quantity"),
        try_to_timestamp(col("InvoiceDate"), lit("M/d/yyyy H:mm")).alias("invoice_date"),
        col("UnitPrice").cast("double").alias("unit_price"),
        col("CustomerID").cast("string").alias("customer_id"),
        col("Country").cast("string").alias("country"),
        col("_load_time")
      )
    )

    delta_table = DeltaTable.forName(spark, "ecommerce.silver.invoicesSpark")
    delta_table.alias("target").merge(
      source_df.alias("source"),
      "target.invoice_no = source.invoice_no AND target.stock_code = source.stock_code AND target.customer_id = source.customer_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("Update completed.")


# Step 3: Gold Layer
Create reports using the trated data.

In [0]:
# Python/Spark implementation of the Gold Layer - Total Sales per Country, Monthly Sales Summary, Total Per Product

from pyspark.sql.functions import col, expr, try_to_timestamp, lit, sum, count, day, month, year, round
from pyspark.sql.window import Window
from delta.tables import DeltaTable

silver_df = spark.table('ecommerce.silver.invoicesSpark')

dailyReports = {
    "salesPerCountry" : (
            silver_df
            .groupBy(col("country"))
            .agg(
                sum(col("quantity")).alias("total_quantity"),
                count(col("invoice_no")).alias("total_invoices"),
                round(sum(col("quantity") * col("unit_price")),2).alias("total_revenue"),
                expr("date_format(max(invoice_date), 'MM/dd/yyyy HH:mm')").alias("last_purchase_date")
            )
            .withColumn(
                "market_share",
                round(col("total_revenue") / sum(col("total_revenue")).over(Window.partitionBy()) * lit(100),3)
            )
            .orderBy(col("market_share").desc())
        ),
    "monthlySalesSummary" : (
            silver_df
            .groupBy(expr("date_format(invoice_date, 'yyyyMM')").alias('year_month'),expr("date_format(invoice_date, 'yyyy')").alias('year'),expr("date_format(invoice_date, 'MM')").alias('month'))
            .agg(
                sum(col("quantity")).alias("total_quantity"),
                count(col("invoice_no")).alias("total_invoices"),
                round(sum(col("quantity") * col("unit_price")),2).alias("total_revenue"),
                expr("date_format(max(invoice_date), 'MM/dd/yyyy HH:mm')").alias("last_purchase_date")
            )
            .withColumn(
                "market_share",
                round(col("total_revenue") / sum(col("total_revenue")).over(Window.partitionBy()) * lit(100), 3)
            )
            .orderBy(col("year_month").desc())
        ),
    "totalPerProduct" : (
            silver_df
            .groupBy(col("stock_code"), col("description"))
            .agg(
                sum(col("quantity")).alias("total_quantity"),
                count(col("invoice_no")).alias("total_invoices"),
                round(sum(col("quantity") * col("unit_price")),2).alias("total_revenue"),
                expr("date_format(max(invoice_date), 'MM/dd/yyyy HH:mm')").alias("last_purchase_date")
            )
            .withColumn(
                "market_share",
                round(col("total_revenue") / sum(col("total_revenue")).over(Window.partitionBy()) * lit(100), 3)
            )
            .orderBy(col("market_share").desc())
        )
}

for reportName, df in dailyReports.items():
    print(f'Writing table {reportName}')
    (
        df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"ecommerce.gold.{reportName}")
    )

In [0]:
silver_df.printSchema()

In [0]:
%sql
select * from ecommerce.gold.salespercountry;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select * from ecommerce.gold.monthlysalessummary;


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select * from ecommerce.gold.totalperproduct;

Databricks visualization. Run in Databricks to view.